# HNeRV (16×3×4 embedding) — RGB ön eğitim + FP32 task döngüsü

**Sıra:** Kurulum → Audit → A1–A4 (HNeRV RGB aşaması) → Hücre 1–10 (task aşaması, v6 döngüsü, FP32 seçim).

Colab yeniden başlarsa: A1 yeni bir klasör açar. Bu durumda Hücre 1'deki `source_path5`'e A4'ün yazdırdığı yolu elle yaz.

## Kurulum

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!uv pip -q install av segmentation-models-pytorch timm einops safetensors brotli

In [3]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = Path("/content/drive/MyDrive/deneme/models")
TARGET_CACHE = Path("/content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt")
NUM_PAIRS = 600

for path in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE):
    assert path.exists(), path

sys.path.insert(0, "/content/drive/MyDrive/deneme")
from research.modules import DistortionNet
from research.oracle_pair import pose_preprocess_grad, seg_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(
    MODEL_DIR / "posenet.safetensors",
    MODEL_DIR / "segnet.safetensors",
    device,
)
distortion_net.requires_grad_(False)

targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets_gpu = targets["seg_class"].to(device, dtype=torch.uint8)
pose_targets_gpu = targets["pose"].to(device, dtype=torch.float32)
assert seg_targets_gpu.shape[0] == pose_targets_gpu.shape[0] == NUM_PAIRS

print("GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "yok")

GPU: Tesla T4


In [4]:
def inflate_for_training(native):
    b, t, c, h, w = native.shape
    assert (t, c, h, w) == (2, 3, 384, 512), native.shape

    flat = native.reshape(b * 2, 3, h, w)
    up = F.interpolate(flat, size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)

    # İleri geçişte gerçek çözücü gibi yuvarlar; geri yayılımda gradyan geçer.
    rounded = up + (up.round() - up).detach()
    return rounded.reshape(b, 2, 3, 874, 1164)


@torch.inference_mode()
def measure_decoder(eval_batch_size=8):
    pose_sum = 0.0
    seg_sum = 0.0
    pair_count = 0

    flags = [(m, m.training) for root in (decoder, distortion_net) for m in root.modules()]
    old_mm = torch.backends.cuda.matmul.allow_tf32
    old_cudnn = torch.backends.cudnn.allow_tf32

    decoder.eval()
    distortion_net.eval()
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    try:
        with torch.autocast(device_type=torch.device(device).type, enabled=False):
            for start in range(0, NUM_PAIRS, eval_batch_size):
                ids = torch.arange(start, min(start + eval_batch_size, NUM_PAIRS), device=device)
                candidate = inflate_for_training(decoder(latents[ids]))

                pred_pose = distortion_net.posenet(
                    pose_preprocess_grad(candidate).contiguous(memory_format=torch.channels_last)
                )["pose"][:, :6].float()
                pred_seg = distortion_net.segnet(
                    seg_preprocess_grad(candidate).contiguous(memory_format=torch.channels_last)
                ).argmax(dim=1)

                pose_error = (pred_pose - pose_targets_gpu[ids].float()).square().double().mean(dim=1)
                seg_error = (pred_seg != seg_targets_gpu[ids].long()).double().mean(dim=(1, 2))

                pose_sum += pose_error.sum().item()
                seg_sum += seg_error.sum().item()
                pair_count += len(ids)

        pose_dist = pose_sum / pair_count
        seg_dist = seg_sum / pair_count
        score = 100 * seg_dist + (10 * pose_dist) ** 0.5
        return pose_dist, seg_dist, score

    finally:
        torch.backends.cuda.matmul.allow_tf32 = old_mm
        torch.backends.cudnn.allow_tf32 = old_cudnn
        for module, training in flags:
            module.training = training

In [5]:
import importlib.util
import urllib.request

CODEC_PATH = Path("/content/hnerv_codec.py")
CODEC_URL = (
    "https://raw.githubusercontent.com/commaai/"
    "comma_video_compression_challenge/"
    "20fbc0d3c4dfc1262fc3a3509e7d69d77fefadc2/"
    "submissions/hnerv_muon/src/codec.py"
)
if not CODEC_PATH.exists():
    urllib.request.urlretrieve(CODEC_URL, CODEC_PATH)

spec = importlib.util.spec_from_file_location("hnerv_codec", CODEC_PATH)
hnerv_codec = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hnerv_codec)

## Audit fonksiyonları
Sadece fonksiyon tanımlar, çalıştırınca bir şey yazdırmaz.

In [6]:
import csv
import json
import math
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F


# ------------------------------------------------------------------
# AUDIT
# ------------------------------------------------------------------

def interface_bands5(truth):
    is0, is1 = truth == 0, truth == 1

    near1 = F.max_pool2d(
        is1.float().unsqueeze(1), 3, 1, 1
    )[:, 0] > 0
    near0 = F.max_pool2d(
        is0.float().unsqueeze(1), 3, 1, 1
    )[:, 0] > 0

    interface = (is0 & near1) | (is1 & near0)

    near3 = F.max_pool2d(
        interface.float().unsqueeze(1), 7, 1, 3
    )[:, 0] > 0
    near6 = F.max_pool2d(
        interface.float().unsqueeze(1), 13, 1, 6
    )[:, 0] > 0

    return near3, near6 & ~near3, ~near6


def summarize_audit5(pair_confusion, pose_sq, band_confusion, frame_ids):
    conf = pair_confusion.sum(0)
    support = conf.sum(1)
    wrong = support - conf.diag()
    false_positive = conf.sum(0) - conf.diag()

    pixels_per_pair = pair_confusion.sum((1, 2))
    pair_wrong = (
        pixels_per_pair
        - pair_confusion.diagonal(dim1=1, dim2=2).sum(1)
    )
    pair_seg = pair_wrong.double() / pixels_per_pair
    pair_pose = pose_sq.double().mean(1)

    n = len(pair_seg)
    seg = float(pair_seg.mean())
    pose = float(pair_pose.mean())

    seg_score = 100 * seg
    pose_score = math.sqrt(10 * pose)

    # Yalnız bu pair'in hataları sıfır olsaydı teorik skor kazancı.
    oracle_gain = (
        100 * pair_seg / n
        + pose_score
        - (10 * (pose - pair_pose / n).clamp_min(0)).sqrt()
    )

    total_pixels = int(conf.sum())
    total_wrong = int(wrong.sum())

    classes = []
    for c in range(len(conf)):
        count = int(support[c])
        err = int(wrong[c])

        classes.append({
            "class_id": c,
            "support": count,
            "wrong": err,
            "error_pct": 100 * err / count if count else None,
            "false_positive": int(false_positive[c]),
            "share_of_all_errors_pct": (
                100 * err / total_wrong if total_wrong else 0.0
            ),
            "seg_score_contribution": 100 * err / total_pixels,
        })

    bands = []
    for name, cm in zip(("0-3px", "4-6px", ">6px"), band_confusion):
        count = int(cm.sum())
        err = count - int(cm.diag().sum())
        d0 = int(cm[0].sum())
        d1 = int(cm[1].sum())

        bands.append({
            "band": name,
            "pixels": count,
            "wrong_all_classes": err,
            "error_pct": 100 * err / count if count else None,
            "share_of_all_errors_pct": (
                100 * err / total_wrong if total_wrong else 0.0
            ),
            "n01": int(cm[0, 1]),
            "n10": int(cm[1, 0]),
            "share_of_all_01_pct": (
                100 * int(cm[0, 1]) / int(conf[0, 1])
                if int(conf[0, 1]) else 0.0
            ),
            "share_of_all_10_pct": (
                100 * int(cm[1, 0]) / int(conf[1, 0])
                if int(conf[1, 0]) else 0.0
            ),
            "class0_support": d0,
            "class1_support": d1,
            "rate01_pct": 100 * int(cm[0, 1]) / d0 if d0 else None,
            "rate10_pct": 100 * int(cm[1, 0]) / d1 if d1 else None,
        })

    pairs = []
    for i in range(n):
        cm = pair_confusion[i]
        bad = cm.clone()
        bad.fill_diagonal_(0)
        dominant = int(bad.argmax())

        row = {
            "pair_id": i,
            "frame0": int(frame_ids[i, 0]),
            "frame1": int(frame_ids[i, 1]),
            "seg_frame": int(frame_ids[i, 1]),
            "wrong_pixels": int(pair_wrong[i]),
            "seg_error_pct": 100 * float(pair_seg[i]),
            "seg_dist": float(pair_seg[i]),
            "pose_mse": float(pair_pose[i]),
            "n01": int(cm[0, 1]),
            "n10": int(cm[1, 0]),
            "dominant_confusion": (
                f"{dominant // len(conf)}->{dominant % len(conf)}"
                if int(bad.sum()) else "none"
            ),
            "oracle_gain_if_pair_perfect": float(oracle_gain[i]),
        }

        for c in range(len(conf)):
            row[f"class{c}_support"] = int(cm[c].sum())
            row[f"class{c}_wrong"] = int(cm[c].sum() - cm[c, c])

        for d in range(pose_sq.shape[1]):
            row[f"pose_dim{d}_squared_error"] = float(pose_sq[i, d])

        pairs.append(row)

    return {
        "pose_dist": pose,
        "seg_dist": seg,
        "distortion_score": seg_score + pose_score,
        "seg_score_term": seg_score,
        "pose_score_term": pose_score,
        "total_wrong": total_wrong,
        "total_pixels": total_pixels,
        "confusion": conf.tolist(),
        "classes": classes,
        "bands": bands,
        "pairs": pairs,
        "pose_dimension_mse": pose_sq.double().mean(0).tolist(),
        "metric": (
            "100*mean_pair_seg_disagreement + "
            "sqrt(10*mean_pair_pose_mse); RATE EXCLUDED"
        ),
        "frame_id_convention": (
            "zero-based; supplied frame_ids mapping; "
            "SegNet uses pair's last frame"
        ),
    }


@torch.inference_mode()
def audit_decoder5(eval_batch_size=None):
    batch_size = (
        EVAL_BATCH5 if eval_batch_size is None else eval_batch_size
    )

    flags = [
        (m, m.training)
        for root in (decoder, distortion_net)
        for m in root.modules()
    ]
    old_mm = torch.backends.cuda.matmul.allow_tf32
    old_cudnn = torch.backends.cudnn.allow_tf32

    decoder.eval()
    distortion_net.eval()
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    pair_cms = []
    pose_errors = []
    total_bands = None
    gallery = []

    try:
        with torch.autocast(
            device_type=torch.device(device).type,
            enabled=False,
        ):
            for start in range(0, NUM_PAIRS, batch_size):
                ids = torch.arange(
                    start,
                    min(start + batch_size, NUM_PAIRS),
                    device=device,
                )

                full = inflate_for_training(decoder(latents[ids]))
                seg_input = seg_preprocess_grad(full).contiguous(
                    memory_format=torch.channels_last
                )

                logits = distortion_net.segnet(seg_input).float()
                pred = logits.argmax(1)
                truth = seg_targets_gpu[ids].long()
                c = logits.shape[1]

                assert truth.shape == pred.shape and c >= 2
                assert bool(((truth >= 0) & (truth < c)).all()), (
                    "Unsupported target/ignore label"
                )

                pose = distortion_net.posenet(
                    pose_preprocess_grad(full).contiguous(
                        memory_format=torch.channels_last
                    )
                )["pose"][:, :6].float()

                sq = (pose - pose_targets_gpu[ids].float()).square()

                if not bool(
                    torch.isfinite(logits).all()
                    & torch.isfinite(sq).all()
                ):
                    raise FloatingPointError(
                        "Non-finite audit outputs; checkpoint not accepted"
                    )

                encoded = truth * c + pred
                offsets = (
                    torch.arange(len(ids), device=device)[:, None, None]
                    * c * c
                )

                cm = torch.bincount(
                    (encoded + offsets).flatten(),
                    minlength=len(ids) * c * c,
                )
                cm = cm.reshape(len(ids), c, c).cpu()

                pair_cms.append(cm)
                pose_errors.append(sq.cpu())

                band_cm = torch.stack([
                    torch.bincount(
                        encoded[mask],
                        minlength=c * c,
                    ).reshape(c, c)
                    for mask in interface_bands5(truth)
                ]).cpu()

                total_bands = (
                    band_cm
                    if total_bands is None
                    else total_bands + band_cm
                )

                if SAVE_MAPS5:
                    wrong = (
                        cm.sum((1, 2))
                        - cm.diagonal(dim1=1, dim2=2).sum(1)
                    )

                    for j, count in enumerate(wrong.tolist()):
                        if (
                            len(gallery) < MAP_TOP5
                            or count > min(x[0] for x in gallery)
                        ):
                            rgb = F.interpolate(
                                full[j:j + 1, -1].float(),
                                size=truth.shape[-2:],
                                mode="bilinear",
                                align_corners=False,
                            )[0]

                            gallery.append((
                                count,
                                start + j,
                                rgb.clamp(0, 255).byte().cpu(),
                                truth[j].byte().cpu(),
                                pred[j].byte().cpu(),
                            ))
                            gallery.sort(key=lambda x: (-x[0], x[1]))
                            gallery = gallery[:MAP_TOP5]

                del full, logits, pred, truth, pose, sq
                del seg_input, encoded, offsets

        pair_cms = torch.cat(pair_cms)
        pose_errors = torch.cat(pose_errors)

        assert int(total_bands.sum()) == int(pair_cms.sum())

        result = summarize_audit5(
            pair_cms,
            pose_errors,
            total_bands,
            FRAME_IDS5,
        )
        result["activation_alpha"] = float(decoder.alpha)
        result["_gallery"] = gallery
        return result

    finally:
        torch.backends.cuda.matmul.allow_tf32 = old_mm
        torch.backends.cudnn.allow_tf32 = old_cudnn

        for module, training in flags:
            module.training = training


def write_audit5(result, label, previous=None, source=None):
    directory = CONT5_DIR / "diagnostics" / label
    directory.mkdir(parents=True, exist_ok=True)

    payload = {
        k: v for k, v in result.items()
        if not k.startswith("_")
    }

    prior_pairs = previous["pairs"] if previous else result["pairs"]
    source_pairs = source["pairs"] if source else result["pairs"]

    for i, row in enumerate(payload["pairs"]):
        row["delta_wrong_vs_previous_same_candidate"] = (
            row["wrong_pixels"] - prior_pairs[i]["wrong_pixels"]
        )
        row["delta_pose_mse_vs_previous_same_candidate"] = (
            row["pose_mse"] - prior_pairs[i]["pose_mse"]
        )
        row["delta_wrong_vs_source"] = (
            row["wrong_pixels"] - source_pairs[i]["wrong_pixels"]
        )
        row["delta_pose_mse_vs_source"] = (
            row["pose_mse"] - source_pairs[i]["pose_mse"]
        )

    with (directory / "audit.json").open("w") as f:
        json.dump(payload, f, indent=2, allow_nan=False)

    for name, rows in (
        ("pairs", payload["pairs"]),
        ("classes", payload["classes"]),
        ("bands", payload["bands"]),
    ):
        with (directory / f"{name}.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

    with (directory / "confusion.csv").open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["truth/pred"] + list(range(len(result["confusion"])))
        )
        writer.writerows(
            [i] + row for i, row in enumerate(result["confusion"])
        )

    ds = (
        result["seg_score_term"] - previous["seg_score_term"]
        if previous else 0.0
    )
    dp = (
        result["pose_score_term"] - previous["pose_score_term"]
        if previous else 0.0
    )

    lines = [
        (
            f"\n[{label}] ALPHA={result['activation_alpha']:.4f} | "
            f"DISTORTION={result['distortion_score']:.8f} (rate haric)"
        ),
        (
            f"  Seg katkisi={result['seg_score_term']:.8f} "
            f"(delta={ds:+.8f}) | "
            f"Pose katkisi={result['pose_score_term']:.8f} "
            f"(delta={dp:+.8f})"
        ),
        (
            f"  Seg yanlis={result['total_wrong']:,}/"
            f"{result['total_pixels']:,} | "
            f"Pose MSE={result['pose_dist']:.9f}"
        ),
        (
            "  Class | destek | yanlis | sinif ici hata% | "
            "tum hatalarda pay% | Seg score katkisi"
        ),
    ]

    for row in sorted(
        result["classes"],
        key=lambda r: -r["wrong"],
    ):
        pct = (
            "N/A"
            if row["error_pct"] is None
            else f"{row['error_pct']:.3f}"
        )
        lines.append(
            f"  {row['class_id']:5d} | {row['support']:9,d} | "
            f"{row['wrong']:7,d} | {pct:>14} | "
            f"{row['share_of_all_errors_pct']:17.2f} | "
            f"{row['seg_score_contribution']:.8f}"
        )

    cm = result["confusion"]
    transitions = sorted(
        [
            (cm[a][b], a, b)
            for a in range(len(cm))
            for b in range(len(cm))
            if a != b
        ],
        reverse=True,
    )

    lines.append(
        "  En buyuk karisikliklar: "
        + ", ".join(
            f"{a}->{b}: {count:,}"
            for count, a, b in transitions[:PRINT_TOP5]
            if count
        )
    )

    if previous:
        changes = sorted(
            [
                (cm[a][b] - previous["confusion"][a][b], a, b)
                for a in range(len(cm))
                for b in range(len(cm))
                if a != b
            ],
            reverse=True,
        )

        lines.append(
            "  Onceki ayni adaya gore artan karisikliklar: "
            + (
                ", ".join(
                    f"{a}->{b}: +{d:,}"
                    for d, a, b in changes[:PRINT_TOP5]
                    if d > 0
                )
                or "yok"
            )
        )

    for band in result["bands"]:
        lines.append(
            f"  {band['band']:>5}: "
            f"tum sinif hatalari={band['wrong_all_classes']:,} "
            f"(toplam hatanin %{band['share_of_all_errors_pct']:.2f}); "
            f"1->0={band['n10']:,}/{band['class1_support']:,}; "
            f"0->1={band['n01']:,}/{band['class0_support']:,}; "
            f"tum 1->0 hatalarinin "
            f"%{band['share_of_all_10_pct']:.2f}'si"
        )

    for title, key in (
        ("En kotu Seg frameleri", "wrong_pixels"),
        ("En kotu Pose pairleri", "pose_mse"),
        ("Pair kusursuz olsaydi teorik kazanc", "oracle_gain_if_pair_perfect"),
    ):
        lines.append("  " + title + ":")

        for row in sorted(
            result["pairs"],
            key=lambda r: -r[key],
        )[:PRINT_TOP5]:
            lines.append(
                f"    pair={row['pair_id']:03d} "
                f"frames=({row['frame0']},{row['frame1']}) "
                f"seg_frame={row['seg_frame']} "
                f"wrong={row['wrong_pixels']:,} "
                f"Seg=%{row['seg_error_pct']:.4f} "
                f"PoseMSE={row['pose_mse']:.8f} "
                f"1->0={row['n10']:,} 0->1={row['n01']:,} "
                f"teorik_kazanc={row['oracle_gain_if_pair_perfect']:.7f}"
            )

    if previous:
        for title, key in (
            ("En cok Seg bozulmasi", "delta_wrong_vs_previous_same_candidate"),
            ("En cok Pose bozulmasi", "delta_pose_mse_vs_previous_same_candidate"),
        ):
            bad = [r for r in result["pairs"] if r[key] > 0]
            lines.append(
                "  " + title + ": "
                + (
                    ", ".join(
                        f"pair {r['pair_id']} ({r[key]:+.7g})"
                        for r in sorted(
                            bad, key=lambda r: -r[key]
                        )[:PRINT_TOP5]
                    )
                    or "yok"
                )
            )

    dims = result["pose_dimension_mse"]
    lines.append(
        "  Pose boyut MSE (anlam isimleri dogrulanmadi): "
        + ", ".join(
            f"d{i}={x:.8f}" for i, x in enumerate(dims)
        )
    )

    text = "\n".join(lines)
    print(text, flush=True)
    (directory / "report.txt").write_text(text + "\n")

    if result.get("_gallery"):
        from PIL import Image, ImageDraw

        palette = torch.tensor(
            [
                [70, 100, 160],
                [245, 195, 65],
                [80, 170, 125],
                [180, 100, 205],
                [225, 120, 80],
            ],
            dtype=torch.uint8,
        )

        for rank, (_, pair, rgb, truth, pred) in enumerate(
            result["_gallery"]
        ):
            h, w = truth.shape
            colors = palette[torch.arange(len(cm)) % len(palette)]

            error = torch.zeros((h, w, 3), dtype=torch.uint8)
            error[truth != pred] = torch.tensor(
                [245, 180, 30], dtype=torch.uint8
            )
            error[(truth == 1) & (pred == 0)] = torch.tensor(
                [240, 55, 65], dtype=torch.uint8
            )
            error[(truth == 0) & (pred == 1)] = torch.tensor(
                [55, 155, 245], dtype=torch.uint8
            )

            canvas = Image.new("RGB", (4 * w, h + 76), "#151922")
            draw = ImageDraw.Draw(canvas)

            panels = [
                rgb.permute(1, 2, 0),
                colors[truth.long()],
                colors[pred.long()],
                error,
            ]
            names = (
                "Decoded RGB",
                "Reference class IDs",
                "Predicted class IDs",
                "Error map",
            )

            for col, (name, panel) in enumerate(zip(names, panels)):
                canvas.paste(
                    Image.fromarray(panel.numpy()),
                    (col * w, 25),
                )
                draw.text((col * w + 6, 6), name, fill="white")

            draw.text(
                (8, h + 32),
                (
                    f"{label} | alpha={result['activation_alpha']:.4f} | "
                    f"pair {pair} | "
                    f"seg frame {int(FRAME_IDS5[pair, 1])} (zero-based)"
                ),
                fill="white",
            )
            draw.text(
                (8, h + 50),
                (
                    "Error: red=1->0, blue=0->1, "
                    "amber=other, black=correct"
                ),
                fill="white",
            )

            path = directory / (
                f"worst_seg_{rank + 1:02d}_pair_{pair:03d}.png"
            )
            canvas.save(path)

            if DISPLAY_MAPS5 and rank == 0 and label.endswith("raw"):
                try:
                    from IPython.display import display
                    display(canvas)
                except ImportError:
                    pass

    result.pop("_gallery", None)
    return result

## A1 — HNeRV mimarisi
- Encoder: pair (6 kanal, 384×512) → ConvNeXt → 16×3×4 embedding. Arşive girmez.
- Decoder: 16×3×4 → 6 blok `Conv(k) → PixelShuffle → GELU` → 384×512 × 2 kare. Kanallar her blokta 1.2'ye bölünür, kernel 1 → 3.

In [7]:
import math, time
from datetime import datetime

HNERV_ARCH = dict(emb_shape=(16, 3, 4), c0=52, ratio=1.2, min_ch=12,
                  strides=(4, 2, 2, 2, 2, 2), ks_min=1, ks_max=3)
ENC_ARCH = dict(emb_ch=16, dim=64, strides=(4, 2, 2, 2, 2, 2))


class LayerNorm2d(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln = nn.LayerNorm(c)

    def forward(self, x):
        return self.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


class ConvNeXtBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.dw = nn.Conv2d(c, c, 7, padding=3, groups=c)
        self.norm = nn.LayerNorm(c)
        self.pw1 = nn.Linear(c, 4 * c)
        self.pw2 = nn.Linear(4 * c, c)

    def forward(self, x):
        h = self.dw(x).permute(0, 2, 3, 1)
        h = self.pw2(F.gelu(self.pw1(self.norm(h))))
        return x + h.permute(0, 3, 1, 2)


class HNeRVEncoder(nn.Module):
    """Pair (2 kare, 384x512) -> 16x3x4 embedding. Arşive girmez."""
    def __init__(self, emb_ch=16, dim=64, strides=(4, 2, 2, 2, 2, 2)):
        super().__init__()
        layers, cin = [], 6
        for s in strides:
            layers += [nn.Conv2d(cin, dim, s, stride=s), LayerNorm2d(dim), ConvNeXtBlock(dim)]
            cin = dim
        self.body = nn.Sequential(*layers)
        self.out = nn.Conv2d(dim, emb_ch, 1)

    def forward(self, pairs):                       # [B, 2, 3, 384, 512], 0-255
        x = pairs.reshape(pairs.shape[0], 6, 384, 512) / 255.0
        return self.out(self.body(x)).flatten(1)    # [B, 192]


class HNeRVDecoder2(nn.Module):
    """16x3x4 embedding -> [B, 2, 3, 384, 512]. Blok: Conv(k) -> PixelShuffle(s) -> GELU."""
    def __init__(self, emb_shape=(16, 3, 4), c0=52, ratio=1.2, min_ch=12,
                 strides=(4, 2, 2, 2, 2, 2), ks_min=1, ks_max=3):
        super().__init__()
        self.alpha = 0.0                             # eski döngüdeki assert'ler için; kullanılmıyor
        self.emb_shape = tuple(emb_shape)
        self.strides = tuple(strides)
        self.blocks = nn.ModuleList()
        cin = self.emb_shape[0]
        for i, s in enumerate(self.strides):
            cout = max(int(c0 / ratio ** i), min_ch)
            k = min(ks_min + 2 * i, ks_max)
            self.blocks.append(nn.Conv2d(cin, cout * s * s, k, padding=k // 2))
            cin = cout
        self.head = nn.Conv2d(cin, 6, 3, padding=1)

    def forward(self, z):                            # z: [B, 192]
        x = z.view(z.shape[0], *self.emb_shape)
        for conv, s in zip(self.blocks, self.strides):
            x = F.gelu(F.pixel_shuffle(conv(x), s))
        x = torch.sigmoid(self.head(x)) * 255.0
        return x.view(x.shape[0], 2, 3, *x.shape[-2:])


HNERV_DIR = Path("/content/drive/MyDrive/deneme/checkpoints/hnerv2") / f"e192_{datetime.now():%Y%m%d_%H%M%S}"
HNERV_DIR.mkdir(parents=True, exist_ok=False)

print("Decoder parametre:", sum(p.numel() for p in HNeRVDecoder2(**HNERV_ARCH).parameters()))
print("Klasör:", HNERV_DIR)

Decoder parametre: 235950
Klasör: /content/drive/MyDrive/deneme/checkpoints/hnerv2/e192_20260930_194450


## A2 — Kareleri 384×512 olarak yükle
İlk seferde Drive'a önbellek yazar (~700 MB), sonra oradan okur.

In [9]:
import av
from research.frame_utils import yuv420_to_rgb

FRAMES_CACHE = Path("/content/drive/MyDrive/deneme/checkpoints/hnerv384_frames384_bilinear_v1.pt")

if FRAMES_CACHE.is_file():
    frames_h = torch.load(FRAMES_CACHE, map_location="cpu", weights_only=True)
else:
    frames_h = torch.empty((NUM_PAIRS, 2, 3, 384, 512), dtype=torch.uint8)
    container = av.open(str(VIDEO_PATH))
    count = 0
    try:
        with torch.no_grad():
            for frame in container.decode(video=0):
                rgb = yuv420_to_rgb(frame).permute(2, 0, 1).float().unsqueeze(0).to(device)
                small = F.interpolate(rgb, size=(384, 512), mode="bilinear", align_corners=False)
                frames_h[count // 2, count % 2] = small.round().clamp(0, 255).byte()[0].cpu()
                count += 1
    finally:
        container.close()
    assert count == 2 * NUM_PAIRS, count
    torch.save(frames_h, FRAMES_CACHE)

frames_h = frames_h.to(device)
print("Kareler:", tuple(frames_h.shape), frames_h.dtype)

Kareler: (600, 2, 3, 384, 512) torch.uint8


## A3 — RGB eğitimi
Encoder + decoder kareleri yeniden üretmeyi öğrenir. Isınma + kosinüs LR. Her 25 epoch'ta PSNR yazar ve `rgb_last.pt` kaydeder.

In [ ]:
H_EPOCHS = 300
H_BATCH = 8
H_LR = 1e-3
H_WARMUP = 0.1          # adımların %10'u ısınma, sonra kosinüs
H_LOG_EVERY = 25

torch.manual_seed(0)
encoder_h = HNeRVEncoder(**ENC_ARCH).to(device)
decoder = HNeRVDecoder2(**HNERV_ARCH).to(device)

opt_h = torch.optim.Adam(list(encoder_h.parameters()) + list(decoder.parameters()), lr=H_LR)
steps_per_epoch_h = math.ceil(NUM_PAIRS / H_BATCH)
total_h = H_EPOCHS * steps_per_epoch_h
warm_h = int(H_WARMUP * total_h)
sched_h = torch.optim.lr_scheduler.LambdaLR(
    opt_h,
    lambda s: (s + 1) / warm_h if s < warm_h
    else 0.5 * (1 + math.cos(math.pi * (s - warm_h) / (total_h - warm_h))),
)

t0_h = time.time()
for epoch_h in range(1, H_EPOCHS + 1):
    encoder_h.train()
    decoder.train()
    order_h = torch.randperm(NUM_PAIRS, device=device)
    mse_sum_h = 0.0

    for start in range(0, NUM_PAIRS, H_BATCH):
        target = frames_h[order_h[start:start + H_BATCH]].float()
        pred = decoder(encoder_h(target))
        loss = F.mse_loss(pred / 255.0, target / 255.0)

        opt_h.zero_grad(set_to_none=True)
        loss.backward()
        opt_h.step()
        sched_h.step()
        mse_sum_h += loss.item() * len(target)

    psnr_h = 10 * math.log10(NUM_PAIRS / mse_sum_h)
    if epoch_h == 1 or epoch_h % H_LOG_EVERY == 0:
        print(f"E{epoch_h:03d} | PSNR={psnr_h:.2f} dB | {time.time() - t0_h:.0f}s", flush=True)
        torch.save({"encoder": encoder_h.state_dict(), "decoder": decoder.state_dict(),
                    "optimizer": opt_h.state_dict(), "epoch": epoch_h},
                   HNERV_DIR / "rgb_last.pt")

## A4 — Embedding'leri çıkar, ölç, task aşaması için kaydet
Bu skorun 0.175'ten kötü çıkması normal: model henüz sadece RGB için eğitildi.

In [ ]:
encoder_h.eval()
decoder.eval()
with torch.no_grad():
    emb_h = torch.cat([encoder_h(frames_h[s:s + 16].float()) for s in range(0, NUM_PAIRS, 16)])

latents = nn.Parameter(emb_h.clone())
pose_h, seg_h, score_h = measure_decoder(eval_batch_size=8)[:3]
print(f"HNeRV RGB | dist={score_h:.6f} | pose={pose_h:.6f} | seg=%{100 * seg_h:.4f}")

torch.save({
    "decoder": {k: v.detach().cpu() for k, v in decoder.state_dict().items()},
    "latents": emb_h.cpu(),
    "activation_alpha": 0.0,
    "arch": HNERV_ARCH,
    "rgb_psnr": psnr_h,
    "distortion_score": score_h,
}, HNERV_DIR / "best.pt")
torch.save(encoder_h.state_dict(), HNERV_DIR / "encoder.pt")
print("Kaynak:", HNERV_DIR / "best.pt")

HNeRV RGB | dist=9.052000 | pose=5.634567 | seg=%1.5456
Kaynak: /content/drive/MyDrive/deneme/checkpoints/hnerv2/e192_20260930_133237/best.pt


In [ ]:
import gc

# Task aşamasında gerekmeyenler: RGB kareleri (~700 MB GPU), encoder ve onun optimizer'ı,
# yarım kalan eğitim adımının ara değişkenleri
for name in ["frames_h", "encoder_h", "opt_h", "sched_h", "emb_h", "target", "pred", "loss",
             "seg_teacher_logits_gpu", "teacher_logits_cpu4", "optimizer5", "scheduler5",
             "ema_decoder5", "ema_latents5", "raw5", "ema_state5"]:
    globals().pop(name, None)

# OOM yarıda kestiyse kalan gradyanları sil
decoder.zero_grad(set_to_none=True)
latents.grad = None

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"GPU kullanımda: {torch.cuda.memory_allocated() / 2**30:.2f} GiB | "
      f"ayrılmış: {torch.cuda.memory_reserved() / 2**30:.2f} GiB")

GPU kullanımda: 0.28 GiB | ayrılmış: 1.44 GiB


## Hücre 1 — Task aşaması ayarları

In [11]:
import csv, io, json, zipfile

TARGET_ALPHA5 = 0.0

EPOCHS5 = 40
BATCH5 = 8
STEPS_PER_EPOCH5 = 75
SAMPLES_PER_EPOCH5 = BATCH5 * STEPS_PER_EPOCH5

LR_DECODER5 = 2e-4
LR_LATENTS5 = 1e-3
WEIGHT_DECAY5 = 0.0

K_FRAC5 = 0.15
CLASS1_BAND3_BOOST5 = 2.0
BOUNDARY01_WEIGHT5 = 0.30

EMA_DECAY5 = 0.99
SCHED_PATIENCE5 = 1      # 1 epoch iyileşme yoksa LR yarıya
PATIENCE5 = 6            # 6 epoch iyileşme yoksa dur

EVAL_BATCH5 = 8
PRINT_TOP5 = 5
SAVE_MAPS5 = True
MAP_TOP5 = 3
DISPLAY_MAPS5 = False

RATE_WEIGHT5 = 25
VIDEO_BYTES5 = VIDEO_PATH.stat().st_size
META5 = {"n_pairs": 600, "latent_dim": 192, "eval_size": [384, 512], "arch": "hnerv2", **HNERV_ARCH}

FRAME_IDS5 = torch.arange(NUM_PAIRS * 2).reshape(NUM_PAIRS, 2)
HNERV_DIR = Path("/content/drive/MyDrive/deneme/checkpoints/hnerv2/e192_20260930_133237/")
source_path5 = HNERV_DIR / "best.pt"      # Colab yeniden başladıysa A4'ün yazdırdığı yolu buraya yaz

assert NUM_PAIRS == 600
assert source_path5.is_file(), source_path5

## Hücre 3 — Aday yükleme, paketleme ve FP32 ölçüm

In [12]:
import contextlib


def snapshot5():
    return (
        {k: v.detach().cpu().clone() for k, v in decoder.state_dict().items()},
        latents.detach().cpu().clone(),
        float(decoder.alpha),
    )


def load_candidate5(decoder_state, latents_cpu, alpha):
    decoder.load_state_dict(decoder_state)
    decoder.alpha = alpha
    with torch.no_grad():
        latents.copy_(latents_cpu.to(device))


def pack5(decoder_state, latents_cpu):
    bin_bytes = hnerv_codec.build_archive(decoder_state, latents_cpu.float().cpu(), META5)

    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        zf.writestr("0.bin", bin_bytes)
    archive_bytes = buffer.getvalue()

    decoder_q, latents_q, _ = hnerv_codec.parse_archive(bin_bytes)
    return decoder_q, latents_q, archive_bytes


def fp32_audit5(decoder_state, latents_cpu, alpha, label, previous=None, source=None):
    live = snapshot5()
    try:
        load_candidate5(decoder_state, latents_cpu, alpha)
        result = audit_decoder5()
    finally:
        load_candidate5(*live)
    with contextlib.redirect_stdout(io.StringIO()):   # rapor diske yazılır, ekrana basılmaz
        result = write_audit5(result, label, previous, source)
    return result


def fmt5(tag, r):
    cls = " ".join(
        f"c{c['class_id']}=%{c['error_pct']:.3f}" if c["error_pct"] is not None else f"c{c['class_id']}=N/A"
        for c in r["classes"]
    )
    return (f"{tag} | dist={r['distortion_score']:.6f} | pose={r['pose_dist']:.6f} "
            f"| seg=%{100 * r['seg_dist']:.4f} | {cls}")

## Hücre 4 — Kaynağı yükle ve ölçümü doğrula

In [16]:
CONT5_DIR = source_path5.parent / f"task_fp32_{datetime.now():%Y%m%d_%H%M%S}"
CONT5_DIR.mkdir(parents=True, exist_ok=False)

source5 = torch.load(source_path5, map_location="cpu", weights_only=True)
SOURCE_ALPHA5 = float(source5.get("activation_alpha", 0.0))

if ("decoder" not in globals() or not isinstance(decoder, HNeRVDecoder2)
        or "latents" not in globals() or latents.shape != source5["latents"].shape):
    decoder = HNeRVDecoder2(**HNERV_ARCH).to(device)
    latents = nn.Parameter(torch.zeros_like(source5["latents"]).to(device))
META5["latent_dim"] = source5["latents"].shape[1]

load_candidate5(source5["decoder"], source5["latents"], SOURCE_ALPHA5)

distortion_net.eval()
distortion_net.requires_grad_(False)

reference5 = measure_decoder(eval_batch_size=EVAL_BATCH5)
source_audit5 = fp32_audit5(source5["decoder"], source5["latents"], SOURCE_ALPHA5, "source_fp32")
for name5, a5, b5 in zip(
    ("Pose", "Seg", "Distortion"),
    reference5[:3],
    (source_audit5["pose_dist"], source_audit5["seg_dist"], source_audit5["distortion_score"]),
):
    assert math.isclose(float(a5), b5, rel_tol=1e-5, abs_tol=1e-7), f"{name5}: {a5} vs {b5}"
print(fmt5("source    ", source_audit5))

decoder.alpha = TARGET_ALPHA5
live5 = snapshot5()
transition_audit5 = source_audit5

config5 = {
    k: v for k, v in globals().items()
    if k.endswith("5") and k.isupper() and isinstance(v, (int, float, str, bool))
}
config5.update(source=str(source_path5), source_alpha=SOURCE_ALPHA5, arch=HNERV_ARCH,
               selection="FP32 distortion", torch=str(torch.__version__))
(CONT5_DIR / "config.json").write_text(json.dumps(config5, indent=2))

source     | dist=9.052000 | pose=5.634567 | seg=%1.5456 | c0=%3.052 c1=%43.554 c2=%0.517 c3=%15.815 c4=%0.510


913

## Hücre 5 — Best takibi

In [17]:
def make_state5(decoder_state, latents_cpu, alpha, audit, label):
    return {
        "decoder": decoder_state,
        "latents": latents_cpu,
        "activation_alpha": alpha,
        "arch": HNERV_ARCH,
        "distortion_score": audit["distortion_score"],   # FP32
        "pose_dist": audit["pose_dist"],
        "seg_dist": audit["seg_dist"],
        "selected_candidate": label,
        "source": str(source_path5),
        "config": config5,
    }


best_state5 = make_state5(source5["decoder"], source5["latents"], SOURCE_ALPHA5, source_audit5, "source")
best_path5 = CONT5_DIR / "best.pt"
torch.save(best_state5, best_path5)

## Hücre 6 — Loss (v6 ile aynı)

In [18]:
def compute_loss5(ids):
    truth = seg_targets_gpu[ids].long()
    full = inflate_for_training(decoder(latents[ids]))

    logits = distortion_net.segnet(seg_preprocess_grad(full)).float()
    pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()

    # global margin + top-k
    correct = logits.gather(1, truth.unsqueeze(1))
    others = logits.clone()
    others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
    per_pixel = 0.3 * F.softplus(-(correct - others.amax(1, keepdim=True)) / 0.3)
    flat = per_pixel.flatten()
    topk, _ = flat.topk(max(1, int(K_FRAC5 * flat.numel())))
    margin_loss = 0.6 * topk.mean() + 0.4 * flat.mean()

    # 0/1 arayüzü, 0–3px bandı
    is0, is1 = truth == 0, truth == 1
    band_mask = (is0 | is1) & interface_bands5(truth)[0]
    pair_margin = torch.where(is1, logits[:, 1] - logits[:, 0], logits[:, 0] - logits[:, 1])
    weights = band_mask.float() * (1.0 + (CLASS1_BAND3_BOOST5 - 1.0) * is1.float())
    boundary_loss = (0.3 * F.softplus(-pair_margin / 0.3) * weights).mean()

    ce = F.cross_entropy(logits, truth)
    seg_loss = margin_loss + 0.1 * ce + BOUNDARY01_WEIGHT5 * boundary_loss
    pose_loss = torch.sqrt(10 * F.mse_loss(pose, pose_targets_gpu[ids].float()) + 1e-6)

    loss = 100 * seg_loss + pose_loss
    parts = {
        "margin": 100 * margin_loss.item(),
        "ce": 10 * ce.item(),
        "boundary": 100 * BOUNDARY01_WEIGHT5 * boundary_loss.item(),
        "pose": pose_loss.item(),
        "band_fraction": band_mask.float().mean().item(),
    }
    return loss, parts

## Hücre 7 — Optimizer, scheduler, EMA

In [19]:
torch.manual_seed(20260927)
load_candidate5(best_state5["decoder"], best_state5["latents"], TARGET_ALPHA5)

optimizer5 = torch.optim.AdamW([
    {"params": decoder.parameters(), "lr": LR_DECODER5, "weight_decay": WEIGHT_DECAY5},
    {"params": [latents], "lr": LR_LATENTS5, "weight_decay": 0.0},
])
scheduler5 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer5, mode="min", factor=0.5, patience=SCHED_PATIENCE5,
    threshold=1e-5, threshold_mode="abs",
    min_lr=[LR_DECODER5 / 100, LR_LATENTS5 / 100],
)

ema_decoder5 = {k: v.detach().clone() for k, v in decoder.state_dict().items()}
ema_latents5 = latents.detach().clone()


@torch.no_grad()
def update_ema5():
    for k, v in decoder.state_dict().items():
        if v.is_floating_point():
            ema_decoder5[k].mul_(EMA_DECAY5).add_(v, alpha=1 - EMA_DECAY5)
        else:
            ema_decoder5[k].copy_(v)
    ema_latents5.mul_(EMA_DECAY5).add_(latents, alpha=1 - EMA_DECAY5)

## Hücre 8 — Bir epoch eğitim

In [20]:
def train_epoch5():
    decoder.train()
    assert float(decoder.alpha) == TARGET_ALPHA5

    repeats = math.ceil(SAMPLES_PER_EPOCH5 / NUM_PAIRS)
    order = torch.cat([torch.randperm(NUM_PAIRS, device=device) for _ in range(repeats)])
    order = order[:SAMPLES_PER_EPOCH5]

    sums = {}
    for start in range(0, SAMPLES_PER_EPOCH5, BATCH5):
        optimizer5.zero_grad(set_to_none=True)
        loss, parts = compute_loss5(order[start:start + BATCH5])

        if not torch.isfinite(loss):
            raise FloatingPointError("Loss NaN/Inf; best.pt korunuyor")
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            list(decoder.parameters()) + [latents], 1.0, error_if_nonfinite=True
        )
        optimizer5.step()
        update_ema5()

        parts.update(loss=loss.item(), grad_norm=float(grad_norm), clipped=float(grad_norm > 1.0))
        for k, v in parts.items():
            sums[k] = sums.get(k, 0.0) + v

    return {k: v / STEPS_PER_EPOCH5 for k, v in sums.items()}

## Hücre 9 — Ana döngü (FP32 seçim)

In [21]:
previous_raw5 = previous_ema5 = source_audit5
epochs_since_best5 = 0

for epoch5 in range(1, EPOCHS5 + 1):
    started5 = time.time()
    lr_used5 = [g["lr"] for g in optimizer5.param_groups]

    training5 = train_epoch5()

    raw5 = snapshot5()
    ema_state5 = ({k: v.cpu().clone() for k, v in ema_decoder5.items()}, ema_latents5.cpu().clone())
    raw_audit5 = fp32_audit5(raw5[0], raw5[1], TARGET_ALPHA5,
                             f"epoch_{epoch5:02d}_raw", previous_raw5, source_audit5)
    ema_audit5 = fp32_audit5(ema_state5[0], ema_state5[1], TARGET_ALPHA5,
                             f"epoch_{epoch5:02d}_ema", previous_ema5, source_audit5)
    previous_raw5, previous_ema5 = raw_audit5, ema_audit5

    scheduler5.step(raw_audit5["distortion_score"])

    improved5 = False
    for label5, (dec5, lat5), audit5 in (
        (f"epoch_{epoch5:02d}_raw", raw5[:2], raw_audit5),
        (f"epoch_{epoch5:02d}_ema", ema_state5, ema_audit5),
    ):
        if audit5["distortion_score"] < best_state5["distortion_score"]:
            best_state5 = make_state5(dec5, lat5, TARGET_ALPHA5, audit5, label5)
            improved5 = True
    if improved5:
        torch.save(best_state5, best_path5)
    epochs_since_best5 = 0 if improved5 else epochs_since_best5 + 1

    torch.save({
        "decoder": raw5[0], "latents": raw5[1], "activation_alpha": TARGET_ALPHA5, "arch": HNERV_ARCH,
        "optimizer": optimizer5.state_dict(), "scheduler": scheduler5.state_dict(),
        "ema_decoder": ema_state5[0], "ema_latents": ema_state5[1],
        "epoch": epoch5, "best_distortion": best_state5["distortion_score"],
        "rng_cpu": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "config": config5,
    }, CONT5_DIR / "last_training_state.pt")

    row5 = {
        "epoch": epoch5,
        "fp32_raw": raw_audit5["distortion_score"],
        "fp32_ema": ema_audit5["distortion_score"],
        "pose_raw": raw_audit5["pose_dist"], "seg_raw": raw_audit5["seg_dist"],
        "pose_ema": ema_audit5["pose_dist"], "seg_ema": ema_audit5["seg_dist"],
        "best_distortion": best_state5["distortion_score"],
        "best": best_state5["selected_candidate"],
        "lr_decoder": lr_used5[0], "lr_latents": lr_used5[1],
        "seconds": round(time.time() - started5),
        **{f"train_{k}": v for k, v in training5.items()},
    }
    with (CONT5_DIR / "history.csv").open("a", newline="") as f:
        writer5 = csv.DictWriter(f, fieldnames=list(row5))
        if epoch5 == 1:
            writer5.writeheader()
        writer5.writerow(row5)

    print(fmt5(f"E{epoch5:02d} RAW", raw_audit5), flush=True)
    print(fmt5(f"E{epoch5:02d} EMA", ema_audit5), flush=True)

    if epochs_since_best5 >= PATIENCE5:
        break

E01 RAW | dist=3.159620 | pose=0.374115 | seg=%1.2254 | c0=%2.231 c1=%47.322 c2=%0.489 c3=%8.561 c4=%0.321
E01 EMA | dist=4.020475 | pose=0.744894 | seg=%1.2912 | c0=%2.540 c1=%42.419 c2=%0.459 c3=%10.438 c4=%0.377
E02 RAW | dist=2.251760 | pose=0.138192 | seg=%1.0762 | c0=%1.907 c1=%47.476 c2=%0.327 c3=%7.795 c4=%0.381
E02 EMA | dist=2.729668 | pose=0.257275 | seg=%1.1257 | c0=%2.169 c1=%43.971 c2=%0.368 c3=%8.059 c4=%0.323
E03 RAW | dist=2.100726 | pose=0.114628 | seg=%1.0301 | c0=%1.980 c1=%46.042 c2=%0.224 c3=%8.041 c4=%0.354
E03 EMA | dist=2.081516 | pose=0.111505 | seg=%1.0256 | c0=%1.922 c1=%44.846 c2=%0.306 c3=%7.218 c4=%0.297
E04 RAW | dist=1.984534 | pose=0.102934 | seg=%0.9700 | c0=%1.839 c1=%46.796 c2=%0.252 c3=%6.616 c4=%0.243
E04 EMA | dist=1.770094 | pose=0.066151 | seg=%0.9568 | c0=%1.748 c1=%45.137 c2=%0.267 c3=%6.570 c4=%0.287
E05 RAW | dist=1.890317 | pose=0.091511 | seg=%0.9337 | c0=%1.745 c1=%44.014 c2=%0.212 c3=%6.986 c4=%0.311
E05 EMA | dist=1.541044 | pose=0.040

## Hücre 10 — Best'i geri yükle, doğrula ve arşivi yaz

In [ ]:
best_state5 = torch.load(best_path5, map_location="cpu", weights_only=True)
final_audit5 = fp32_audit5(best_state5["decoder"], best_state5["latents"],
                           best_state5["activation_alpha"], "selected_best_fp32", source=source_audit5)
assert math.isclose(final_audit5["distortion_score"], best_state5["distortion_score"], rel_tol=1e-5, abs_tol=1e-7)

load_candidate5(best_state5["decoder"], best_state5["latents"], best_state5["activation_alpha"])
_, _, final_archive5 = pack5(best_state5["decoder"], best_state5["latents"])
(CONT5_DIR / "archive.zip").write_bytes(final_archive5)

print(fmt5("BEST", final_audit5))
print("best.pt:", best_path5)